# Tutorial: Output Inspection and Convergence Diagnostics

Detailed step-by-step workflow notebook for this SD-dMFA repository.


## Audience, Prerequisites, Outcomes

**Audience**
- Analysts diagnosing model behavior and coupling stability.

**Prerequisites**
- Python 3.11+ environment for this repo.
- `pip install -e ".[dev]"` completed.
- Notebook executed from repository root or a subfolder.

**Outcomes**
- Locate latest run artifacts and load key tables.
- Inspect convergence/iteration diagnostics by slice.
- Plot key time series for stress, service, and bottlenecks.


## Outline

1. Find latest run directory
2. Inspect summary and scalar metrics
3. Inspect timeseries indicators
4. Inspect convergence trace tables
5. Plot targeted diagnostics


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path
from textwrap import dedent

import pandas as pd
import matplotlib.pyplot as plt

try:
    from crm_model.common.io import load_run_config
except Exception:
    load_run_config = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 240)


In [ ]:
DRY_RUN = False
RUN_HEAVY = False
RUN_PLOTS = False
RUN_CALIBRATION = False
RUN_AUDIT = False

CONFIG = "configs/runs/mvp.yml"
EXAMPLE_VARIANT = "baseline"


In [ ]:
def find_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "configs").exists() and (cand / "src").exists():
            return cand
    raise RuntimeError("Could not locate repo root from current working directory.")


def sh(cmd: str, *, cwd: Path, check: bool = True) -> subprocess.CompletedProcess | None:
    print(f"$ {cmd}")
    if DRY_RUN:
        return None
    cp = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, capture_output=True)
    if cp.stdout.strip():
        print(cp.stdout)
    if cp.stderr.strip():
        print(cp.stderr)
    if check and cp.returncode != 0:
        raise RuntimeError(f"Command failed ({cp.returncode}): {cmd}")
    return cp


def latest_dir(base: Path) -> Path | None:
    if not base.exists():
        return None
    cands = [p for p in base.iterdir() if p.is_dir() and p.name != "_archive"]
    return sorted(cands)[-1] if cands else None


def load_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"Missing: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


REPO = find_repo_root()
CONFIG_PATH = (REPO / CONFIG).resolve()
CONFIG_STEM = CONFIG_PATH.stem
print("Repo:", REPO)
print("Config:", CONFIG_PATH)


## Step 1: Locate latest run for selected variant


In [ ]:
run_base = REPO / "outputs" / "runs" / CONFIG_STEM / EXAMPLE_VARIANT
latest = latest_dir(run_base)
print("latest:", latest)


## Step 2: Load principal output tables


In [ ]:
if latest is None:
    raise RuntimeError("No run outputs found. Execute notebook 04 first.")
summary = load_csv(latest / "summary.csv")
scalar = load_csv(latest / "indicators" / "scalar_metrics.csv")
ts = load_csv(latest / "indicators" / "timeseries.csv")
conv_it = load_csv(latest / "indicators" / "coupling_convergence_iteration.csv")
conv_y = load_csv(latest / "indicators" / "coupling_signals_iteration_year.csv")

print(len(summary), len(scalar), len(ts), len(conv_it), len(conv_y))


## Step 3: Slice summary table


In [ ]:
show_cols = [
    "material", "region", "iterations", "coupling_converged", "coupling_convergence_metric",
    "final_stress_multiplier", "final_service_stress_signal", "final_bottleneck_pressure_mean",
    "final_collection_rate_mean",
]
show = [c for c in show_cols if c in summary.columns]
display(summary[show].sort_values(["material", "region"]).reset_index(drop=True))


## Step 4: Scalar resilience indicators


In [ ]:
if not scalar.empty:
    pivot = scalar.pivot_table(index=["material", "region"], columns="metric", values="value", aggfunc="last")
    display(pivot.head(20))


## Step 5: Targeted time-series extraction


In [ ]:
if ts.empty:
    raise RuntimeError("timeseries.csv missing")
sel = ts[(ts["material"] == "nickel") & (ts["region"] == "EU27")]
inds = ["Service_level", "Unmet_service", "Coupling_stress_multiplier", "SD_bottleneck_pressure"]
for ind in inds:
    d = sel[sel["indicator"] == ind].sort_values("year")
    print(ind, "rows=", len(d), "min/max=", (d["value"].min() if len(d) else None), (d["value"].max() if len(d) else None))


## Step 6: Plot convergence and bottleneck traces


In [ ]:
if not conv_it.empty:
    fig, ax = plt.subplots(figsize=(7, 4))
    for (m, r), d in conv_it.groupby(["material", "region"]):
        ax.plot(d["iteration"], d["convergence_metric"], marker="o", linewidth=1.2, label=f"{m}-{r}")
    ax.axhline(y=0.02, color="red", linestyle="--", linewidth=1.0, label="tol=0.02")
    ax.set_title("Convergence metric by iteration")
    ax.set_xlabel("iteration")
    ax.set_ylabel("convergence_metric")
    ax.grid(alpha=0.25)
    ax.legend(ncol=3, fontsize=8)
    plt.tight_layout()
    plt.show()


## Pitfalls

- Reading only scalar metrics can hide temporal instability.
- Always check both `coupling_converged` and `iterations` near max_iter for borderline runs.


## Exercises

1. Repeat this workflow with `CONFIG=configs/runs/r-strategies.yml`.
2. Record one thing that changed and why.
3. Add one guardrail/check specific to your team workflow.


In [ ]:
# Exercise answer scaffold
pass
